# 《诸界锁仙》 96GB 满血版 · 电影感多人精品有声剧全自动生产线 (`v1.1.0`)
### 黄金三件套：`Fish Audio Drama 3`（24位原生国风声线克隆与表演） + `MOSS-SoundEffect v2.0`（48kHz 真实拟音 + 6大跨集母题直通） + `ACE-Step 1.5 XL`（48kHz 分幕配乐）

- **为什么本升级版能逼近 ElevenLabs 电影感成片？**
  1. **24 位精调国风干音底座**：直接内置《诸界锁仙》已锁定的全剧 24 位纯正大陆普通话戏剧干音（`44.1kHz` `-16 LUFS` WAV + 100% 逐字对齐文本），零 API 费用继承高辨识度角色声优！
  2. **Drama 3 原生导演指令 + 70+ 仙侠多音字纠偏**：自动转译情绪描述、`[cold laugh]`/`[sighs]` 气口、`<whisper>…</whisper>` 成对韵律标签与古账册/境界多音字。
  3. **SGLang-Omni `RAS` 高表现力采样 + CPU `int8` `faster-whisper` 零显存质检门**：自动拦截标签漏读与句尾吞字，首尾死寂智能微切边。
  4. **完整三母线 `-16 LUFS` 母带混音 (`aigc-audio-pipeline`)**：6 大标志性声效母题直接复用母带 WAV，`overlay_next` 动效叠化不闪避、`mental_realm`/`memory_echo` 识海空间混响、BGM 侧链闪避与中文角色名 `.srt` 字幕。
- **独立安全命名 (`_drama3_master_v001`)**：绝不覆盖前 6 集 ElevenLabs 成片，支持先跑 `AUDITION,E001` 同集盲听对比，再一键连跑 `E007-E040` 全量收官！


## 1. 检测 96GB GPU、挂载 Google Drive 并秒级解压 164MB 工程包至本地 NVMe


In [ ]:
import os, shutil, zipfile
from pathlib import Path
import torch

!nvidia-smi
assert torch.cuda.is_available(), '未检测到 CUDA GPU！请连接 96GB G4 或 A100 运行时后重试。'
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
major, minor = torch.cuda.get_device_capability(0)
print(f'✅ GPU: {torch.cuda.get_device_name(0)} | sm_{major}{minor} | 显存: {vram_gb:.1f} GB')

if os.path.isdir('/content/drive/My Drive') and not os.path.exists('/content/drive/MyDrive'):
    os.symlink('/content/drive/My Drive', '/content/drive/MyDrive')
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

PIPE_CANDS = list(Path('/content/drive/MyDrive').glob('*诸界锁仙*/04_有声剧与广播剧音频/05_Colab_96GB_FishDrama3*'))
assert PIPE_CANDS, '未在 Google Drive 找到《诸界锁仙》/04_有声剧与广播剧音频/05_Colab_96GB_FishDrama3_满血本地推理管线！'
DRIVE_PIPE_DIR = PIPE_CANDS[0].resolve()
DRIVE_OUT_DIR = DRIVE_PIPE_DIR / 'outputs'
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_STUDIO = Path('/content/zhujie_suoxian_work/zhujie_studio')
if not (LOCAL_STUDIO / 'voice_cast_drama3.json').exists():
    LOCAL_STUDIO.mkdir(parents=True, exist_ok=True)
    bundle_zip = DRIVE_PIPE_DIR / 'zhujie_suoxian_96gb_drama3_bundle.zip'
    print('⏳ 正在将 164MB 自包含工程包解压至本地高速 NVMe:', bundle_zip)
    with zipfile.ZipFile(bundle_zip, 'r') as zf:
        zf.extractall(LOCAL_STUDIO)
print('📂 本地 NVMe 工程就绪:', LOCAL_STUDIO)
print('☁️ 云盘成品自动同步目录:', DRIVE_OUT_DIR)


## 2. 一键准备三路隔离推理环境（自动复用云盘 `AudioStudio_G4_Cache` 源码包）


In [ ]:
import subprocess

ENV_ROOT = Path('/content/nine_prison_envs') if (Path('/content/nine_prison_envs/drama3-venv/bin/sgl-omni')).exists() else Path('/content/audio_studio_envs')
ENV_ROOT.mkdir(parents=True, exist_ok=True)
DRAMA_VENV = ENV_ROOT / 'drama3-venv'
MOSS_VENV = ENV_ROOT / 'moss-venv'
ACESTEP_VENV = ENV_ROOT / 'acestep-venv'
SGL_REPO = ENV_ROOT / 'sglang-omni'
MOSS_REPO = ENV_ROOT / 'MOSS-TTS'
CACHE_TAR_DIR = Path('/content/drive/MyDrive/AI_Workstation/AudioStudio_G4_Cache')

subprocess.check_call(['pip', 'install', '-q', 'uv', 'hf_transfer', 'huggingface_hub>=0.26.0'])

def _venv_ready(vdir: Path, code: str) -> bool:
    py = vdir / 'bin/python'
    return py.exists() and subprocess.run([str(py), '-c', code], capture_output=True).returncode == 0

if not (DRAMA_VENV / 'bin/sgl-omni').exists():
    print('⏳ [1/3] 构建 Drama 3 (SGLang-Omni sm_120) 环境...')
    if not SGL_REPO.exists():
        if (CACHE_TAR_DIR / 'sglang_omni_sm120.tar.gz').exists():
            subprocess.check_call(['tar', '-xzf', str(CACHE_TAR_DIR / 'sglang_omni_sm120.tar.gz'), '-C', str(ENV_ROOT)])
        else:
            subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/sgl-project/sglang-omni.git', str(SGL_REPO)])
    subprocess.check_call(['uv', 'venv', str(DRAMA_VENV), '--python', '3.12', '--allow-existing'])
    subprocess.check_call(['uv', 'pip', 'install', '--python', str(DRAMA_VENV / 'bin/python'), '--prerelease=allow', '-e', '.[fishaudio]', 'hf_transfer'], cwd=SGL_REPO)

if not _venv_ready(MOSS_VENV, 'import moss_soundeffect_v2, scipy, soundfile, faster_whisper'):
    print('⏳ [2/3] 构建 MOSS-SoundEffect v2.0 + CPU faster-whisper 质检环境...')
    if not MOSS_REPO.exists():
        if (CACHE_TAR_DIR / 'moss_tts_v2.tar.gz').exists():
            subprocess.check_call(['tar', '-xzf', str(CACHE_TAR_DIR / 'moss_tts_v2.tar.gz'), '-C', str(ENV_ROOT)])
        else:
            subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/OpenMOSS/MOSS-TTS.git', str(MOSS_REPO)])
    subprocess.check_call(['uv', 'venv', str(MOSS_VENV), '--python', '3.12', '--allow-existing'])
    subprocess.check_call([
        'uv', 'pip', 'install', '--python', str(MOSS_VENV / 'bin/python'),
        '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
        '--index-strategy', 'unsafe-best-match',
        '-e', f'{MOSS_REPO}/moss_soundeffect_v2[torch-cu128]',
        'scipy', 'soundfile', 'requests', 'hf_transfer', 'faster-whisper',
    ])

if not _venv_ready(ACESTEP_VENV, 'from diffusers import AceStepPipeline; import soundfile'):
    print('⏳ [3/3] 构建 ACE-Step 1.5 XL 配乐环境...')
    subprocess.check_call(['uv', 'venv', str(ACESTEP_VENV), '--python', '3.12', '--allow-existing'])
    subprocess.check_call([
        'uv', 'pip', 'install', '--python', str(ACESTEP_VENV / 'bin/python'),
        '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
        '--index-strategy', 'unsafe-best-match',
        'torch==2.9.0+cu128', 'torchaudio==2.9.0+cu128',
        'diffusers>=0.41.0', 'transformers', 'accelerate', 'safetensors', 'packaging',
        'soundfile', 'scipy', 'numpy', 'requests', 'hf_transfer', 'sentencepiece', 'protobuf',
    ])

print('✅ 三路隔离推理环境全部准备就绪！')


## 3. 选择集数并一键启动合成（自动存回 Google Drive `outputs/`）
- **`EPISODES = "AUDITION,E001"`**：推荐首次运行！先花 ~25 秒生成 6 句方案 A 试听合集 + 完整第 1 集，与云盘 `01_声线试听小样` / `03_分集成片音频` 中的 ElevenLabs 版直接盲听对比。
- **`EPISODES = "E007-E040"`**：确认质感满意后，96GB 显存三模型全程常驻，零 API 成本一口气跑完剩余全部 34 章！


In [ ]:
EPISODES = 'AUDITION,E001'   # 可改为 'E007-E040' 或 'E001-E040'
FORCE_INT4 = False           # 96GB G4 默认 False (启用满血 FP8 Drama 3)；40G/80G 显卡会自动切换 int4
SKIP_EXISTING = True         # 已存在成品自动跳过，支持随时断点续跑

cmd = [
    str(MOSS_VENV / 'bin/python'), '-u',
    str(LOCAL_STUDIO / 'run_zhujie_96gb_studio.py'),
    '--studio-dir', str(LOCAL_STUDIO),
    '--episodes', EPISODES,
    '--output-dir', str(DRIVE_OUT_DIR),
    '--cache-dir', str(DRIVE_PIPE_DIR / '.stem_cache'),
    '--sgl-bin', str(DRAMA_VENV / 'bin/sgl-omni'),
    '--moss-python', str(MOSS_VENV / 'bin/python'),
    '--acestep-python', str(ACESTEP_VENV / 'bin/python'),
]
if FORCE_INT4:
    cmd.append('--force-int4')
if SKIP_EXISTING:
    cmd.append('--skip-existing')

env = os.environ.copy()
env['HF_HOME'] = '/content/hf_cache'
env['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
env['TORCHDYNAMO_DISABLE'] = '1'
subprocess.check_call(cmd, env=env)


## 4. 在线试听已生成的 48kHz / -16 LUFS 广播剧成片


In [ ]:
from IPython.display import Audio, display

for mp3 in sorted(DRIVE_OUT_DIR.glob('*.mp3')):
    print(f'🎧 {mp3.name} ({mp3.stat().st_size / 1024 / 1024:.2f} MB)')
    display(Audio(str(mp3)))
